<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/SRCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import h5py
import numpy as np
from pathlib import Path

# Parametreler
TRAINING_DATA_PATH = "/content/drive/MyDrive/2209/high_reso/train/train"
TEST_DATA_PATH = "/content/drive/MyDrive/2209/high_reso/test/val"
RANDOM_CROP_COUNT = 30
INPUT_PATCH_SIZE = 32
OUTPUT_PATCH_SIZE = 20
CONV_BORDER = 6
SCALE_FACTOR = 2

# Blok tabanlı kırpma için parametreler
BLOCK_SIZE = 32
BLOCK_STEP = 16

def prepare_random_samples(image_folder):
    """
    Klasördeki görüntülerden rastgele konumlardan kırpılmış örnekler oluşturur.
    """
    image_files = sorted(os.listdir(image_folder))
    total_samples = len(image_files) * RANDOM_CROP_COUNT

    # Boş dizileri hazırla
    input_data = np.zeros((total_samples, 1, INPUT_PATCH_SIZE, INPUT_PATCH_SIZE), dtype=np.float64)
    target_data = np.zeros((total_samples, 1, OUTPUT_PATCH_SIZE, OUTPUT_PATCH_SIZE), dtype=np.float64)

    for i, filename in enumerate(image_files):
        # Görüntüyü yükle
        image_path = os.path.join(image_folder, filename)
        hr_image = cv2.imread(image_path, cv2.IMREAD_COLOR)

        # BGR'dan YCrCb'ye dönüştür ve sadece Y kanalını al
        hr_image = cv2.cvtColor(hr_image, cv2.COLOR_BGR2YCrCb)[:, :, 0]
        height, width = hr_image.shape

        # Düşük çözünürlüklü görüntü oluştur (orijinali küçült sonra tekrar büyüt)
        lr_image = cv2.resize(hr_image, (width // SCALE_FACTOR, height // SCALE_FACTOR))
        lr_image = cv2.resize(lr_image, (width, height))

        # Rastgele kırpma noktaları oluştur
        max_pos = min(height, width) - INPUT_PATCH_SIZE
        crop_positions_x = np.random.randint(0, max_pos, RANDOM_CROP_COUNT)
        crop_positions_y = np.random.randint(0, max_pos, RANDOM_CROP_COUNT)

        for j in range(RANDOM_CROP_COUNT):
            x, y = crop_positions_x[j], crop_positions_y[j]

            # Giriş ve hedef parçalarını oluştur
            lr_patch = lr_image[x:x + INPUT_PATCH_SIZE, y:y + INPUT_PATCH_SIZE]
            hr_patch = hr_image[x:x + INPUT_PATCH_SIZE, y:y + INPUT_PATCH_SIZE]

            # Normalize et (0-1 aralığına)
            lr_patch = lr_patch.astype(np.float64) / 255.0
            hr_patch = hr_patch.astype(np.float64) / 255.0

            # Veri kümesine ekle
            sample_idx = i * RANDOM_CROP_COUNT + j
            input_data[sample_idx, 0, :, :] = lr_patch
            # Hedef için CONV_BORDER kadar kenarı kırp
            target_data[sample_idx, 0, :, :] = hr_patch[CONV_BORDER:-CONV_BORDER, CONV_BORDER:-CONV_BORDER]

    return input_data, target_data

def prepare_systematic_samples(image_folder):
    """
    Klasördeki görüntülerden sistematik blok tabanlı örnekler oluşturur.
    """
    image_files = sorted(os.listdir(image_folder))

    input_data = []
    target_data = []

    for filename in image_files:
        # Görüntüyü yükle
        image_path = os.path.join(image_folder, filename)
        hr_image = cv2.imread(image_path, cv2.IMREAD_COLOR)

        # BGR'dan YCrCb'ye dönüştür ve sadece Y kanalını al
        hr_image = cv2.cvtColor(hr_image, cv2.COLOR_BGR2YCrCb)[:, :, 0]
        height, width = hr_image.shape

        # Düşük çözünürlüklü görüntü oluştur
        lr_image = cv2.resize(hr_image, (width // SCALE_FACTOR, height // SCALE_FACTOR))
        lr_image = cv2.resize(lr_image, (width, height))

        # Kaç blok oluşturulacağını hesapla
        width_blocks = (height - (BLOCK_SIZE - BLOCK_STEP) * 2) // BLOCK_STEP
        height_blocks = (width - (BLOCK_SIZE - BLOCK_STEP) * 2) // BLOCK_STEP

        # Sistematik parçalama
        for x_idx in range(width_blocks):
            for y_idx in range(height_blocks):
                x = x_idx * BLOCK_STEP
                y = y_idx * BLOCK_STEP

                # Giriş ve hedef parçalarını oluştur
                lr_patch = lr_image[x:x + BLOCK_SIZE, y:y + BLOCK_SIZE]
                hr_patch = hr_image[x:x + BLOCK_SIZE, y:y + BLOCK_SIZE]

                # Normalize et
                lr_patch = lr_patch.astype(np.float64) / 255.0
                hr_patch = hr_patch.astype(np.float64) / 255.0

                # Parçaları uygun formata dönüştür
                lr_sample = np.zeros((1, INPUT_PATCH_SIZE, INPUT_PATCH_SIZE), dtype=np.float64)
                hr_sample = np.zeros((1, OUTPUT_PATCH_SIZE, OUTPUT_PATCH_SIZE), dtype=np.float64)

                lr_sample[0, :, :] = lr_patch
                hr_sample[0, :, :] = hr_patch[CONV_BORDER:-CONV_BORDER, CONV_BORDER:-CONV_BORDER]

                # Veri listelerine ekle
                input_data.append(lr_sample)
                target_data.append(hr_sample)

    # Listeleri numpy dizilerine dönüştür
    input_data = np.array(input_data, dtype=np.float32)
    target_data = np.array(target_data, dtype=np.float32)

    return input_data, target_data

def save_to_h5(input_data, target_data, output_filename):
    """
    Veri kümesini HDF5 formatında kaydeder.
    """
    # Veri türlerini float32'ye dönüştür
    input_data = input_data.astype(np.float32)
    target_data = target_data.astype(np.float32)

    with h5py.File(output_filename, 'w') as h5_file:
        h5_file.create_dataset('data', data=input_data, shape=input_data.shape)
        h5_file.create_dataset('label', data=target_data, shape=target_data.shape)

def load_h5_data(h5_file):
    """
    HDF5 dosyasından veri kümesini yükler.
    """
    with h5py.File(h5_file, 'r') as h5_file:
        input_data = np.array(h5_file.get('data'))
        target_data = np.array(h5_file.get('label'))

        # Kanalları en sona taşı (PyTorch için uygun format)
        input_data = np.transpose(input_data, (0, 2, 3, 1))
        target_data = np.transpose(target_data, (0, 2, 3, 1))

        return input_data, target_data

if __name__ == "__main__":
    # Eğitim verisi için blok tabanlı kırpma kullan
    print("Eğitim verisi hazırlanıyor...")
    train_inputs, train_targets = prepare_systematic_samples(TRAINING_DATA_PATH)
    save_to_h5(train_inputs, train_targets, "train.h5")
    print(f"Eğitim verisi kaydedildi: {train_inputs.shape} giriş, {train_targets.shape} hedef")

    # Test verisi için rastgele kırpma kullan
    print("Test verisi hazırlanıyor...")
    test_inputs, test_targets = prepare_random_samples(TEST_DATA_PATH)
    save_to_h5(test_inputs, test_targets, "test.h5")
    print(f"Test verisi kaydedildi: {test_inputs.shape} giriş, {test_targets.shape} hedef")

Eğitim verisi hazırlanıyor...
Eğitim verisi kaydedildi: (540082, 1, 32, 32) giriş, (540082, 1, 20, 20) hedef
Test verisi hazırlanıyor...
Test verisi kaydedildi: (2130, 1, 32, 32) giriş, (2130, 1, 20, 20) hedef


In [ ]:
import os
import numpy as np
import math
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD, Adam
#from prepare_random_samples import load_h5_data

def create_training_model():
    """
    Eğitim için SRCNN modelini oluşturur.
    Sabit boyutlu giriş şekli (32x32x1) kullanır.
    """
    # SRCNN modeli oluştur
    model = Sequential()

    # İlk evrişim katmanı - Özellik çıkarma
    model.add(Conv2D(
        filters=128,
        kernel_size=(9, 9),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='valid',
        use_bias=True,
        input_shape=(32, 32, 1)
    ))

    # İkinci evrişim katmanı - Haritalama
    model.add(Conv2D(
        filters=64,
        kernel_size=(3, 3),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='same',
        use_bias=True
    ))

    # Üçüncü evrişim katmanı - Yeniden yapılandırma
    model.add(Conv2D(
        filters=1,
        kernel_size=(5, 5),
        kernel_initializer='glorot_uniform',
        activation='linear',
        padding='valid',
        use_bias=True
    ))

    # Modeli derle
    adam_optimizer = Adam(learning_rate=0.0003)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error']
    )

    return model

In [ ]:
def create_prediction_model():
    """
    Tahmin için SRCNN modelini oluşturur.
    Değişken boyutlu giriş şekli (None, None, 1) kullanır.
    """
    # SRCNN modeli oluştur
    model = Sequential()

    # İlk evrişim katmanı - Özellik çıkarma
    model.add(Conv2D(
        filters=128,
        kernel_size=(9, 9),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='valid',
        use_bias=True,
        input_shape=(None, None, 1)
    ))

    # İkinci evrişim katmanı - Haritalama
    model.add(Conv2D(
        filters=64,
        kernel_size=(3, 3),
        kernel_initializer='glorot_uniform',
        activation='relu',
        padding='same',
        use_bias=True
    ))

    # Üçüncü evrişim katmanı - Yeniden yapılandırma
    model.add(Conv2D(
        filters=1,
        kernel_size=(5, 5),
        kernel_initializer='glorot_uniform',
        activation='linear',
        padding='valid',
        use_bias=True
    ))

    # Modeli derle
    adam_optimizer = Adam(learning_rate=0.0003)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error']
    )

    return model

In [ ]:
def calculate_psnr(img1, img2):
    """
    İki görüntü arasındaki PSNR'yi (Peak Signal-to-Noise Ratio) hesaplar.
    """
    mse = np.mean((img1 - img2)**2)
    if mse == 0:
        return float('inf')  # MSE 0 ise PSNR sonsuzdur
    max_pixel = 255.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

In [ ]:
def train_model():
    """
    SRCNN modelini eğitir ve en iyi modeli kaydeder.
    """
    # Eğitim modelini oluştur
    srcnn_model = create_training_model()
    print(srcnn_model.summary())

    # Eğitim ve doğrulama verilerini yükle
    print("Eğitim verilerini yükleme...")
    train_data, train_labels = load_h5_data("/content/train.h5")
    val_data, val_labels = load_h5_data("/content/test.h5")

    # Model kaydetme için callback oluştur
    checkpoint = ModelCheckpoint(
        "SRCNN.h5",
        monitor='val_loss',
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode='min'
    )
    callbacks_list = [checkpoint]

    # Modeli eğit
    print("Model eğitimi başlıyor...")
    srcnn_model.fit(
        train_data, train_labels,
        batch_size=128,
        validation_data=(val_data, val_labels),
        callbacks=callbacks_list,
        shuffle=True,
        epochs=100,
        verbose=1
    )

    print("Eğitim tamamlandı!")

In [ ]:
def predict_image(model_path, image_path, output_folder="/content/drive/MyDrive/2209/high_reso/results"):
    """
    Bir görüntüyü SRCNN ile süper çözünürlüklü hale getirir.

        model_path: Eğitilmiş model ağırlıklarının yolu
        image_path: Girdi görüntüsünün yolu
        output_folder: Sonuçların kaydedileceği klasör
    """
    # Çıktı klasörünü oluştur
    os.makedirs(output_folder, exist_ok=True)

    # Dosya adı bilgilerini hazırla
    base_name = os.path.basename(image_path)
    file_name, _ = os.path.splitext(base_name)
    input_path = os.path.join(output_folder, f"{file_name}_bicubic.png")
    output_path = os.path.join(output_folder, f"{file_name}_srcnn.png")

    # Tahmin modelini oluştur ve ağırlıkları yükle
    srcnn_model = create_prediction_model()
    srcnn_model.load_weights(model_path)
    print(f"Model yüklendi: {model_path}")

    # Orijinal görüntüyü yükle
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # BGR'dan YCrCb'ye dönüştür
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    height, width = img_ycrcb.shape[:2]

    # Y kanalını önce küçült sonra bicubic ile büyüt (düşük çözünürlük simulasyonu)
    y_channel = img_ycrcb[:, :, 0]
    y_channel_lr = cv2.resize(y_channel, (width // 2, height // 2), cv2.INTER_CUBIC)
    y_channel_bicubic = cv2.resize(y_channel_lr, (width, height), cv2.INTER_CUBIC)

    # Bicubic sonucunu kaydet
    img_bicubic = img_ycrcb.copy()
    img_bicubic[:, :, 0] = y_channel_bicubic
    img_bicubic = cv2.cvtColor(img_bicubic, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(input_path, img_bicubic)
    print(f"Bicubic upscaled görüntü kaydedildi: {input_path}")

    # SRCNN için girdiyi hazırla
    input_data = np.zeros((1, height, width, 1), dtype=float)
    input_data[0, :, :, 0] = y_channel_bicubic.astype(float) / 255.0

    # SRCNN tahmini yap
    prediction = srcnn_model.predict(input_data, batch_size=1) * 255.0

    # Değerleri [0, 255] aralığına kırp
    prediction = np.clip(prediction, 0, 255).astype(np.uint8)

    # Tahmini orijinal görüntüye yerleştir (evrişim padding nedeniyle 6 piksel kenarları kırpılır)
    img_srcnn = img_ycrcb.copy()
    img_srcnn[6:-6, 6:-6, 0] = prediction[0, :, :, 0]
    img_srcnn = cv2.cvtColor(img_srcnn, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(output_path, img_srcnn)
    print(f"SRCNN sonucu kaydedildi: {output_path}")

    # PSNR hesaplama
    original_y = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    bicubic_y = cv2.cvtColor(img_bicubic, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    srcnn_y = cv2.cvtColor(img_srcnn, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]

    bicubic_psnr = calculate_psnr(original_y, bicubic_y)
    srcnn_psnr = calculate_psnr(original_y, srcnn_y)

    print(f"Bicubic PSNR: {bicubic_psnr:.2f} dB")
    print(f"SRCNN PSNR: {srcnn_psnr:.2f} dB")
    print(f"PSNR İyileştirmesi: {srcnn_psnr - bicubic_psnr:.2f} dB")

In [ ]:
if __name__ == "__main__":
    # Modeli eğit
    train_model()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)                    │ (None, 24, 24, 128)         │          10,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 24, 24, 64)          │          73,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 20, 20, 1)           │           1,601 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 85,889 (335.50 KB)

 Trainable params: 85,889 (335.50 KB)

 Non-trainable params: 0 (0.00 B)

...

4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 4.3246e-04 - mean_squared_error: 4.3246e-04 - val_loss: 3.6674e-04 - val_mean_squared_error: 3.6674e-04
Epoch 87/100
4213/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.3300e-04 - mean_squared_error: 4.3300e-04
Epoch 87: val_loss did not improve from 0.00037
4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 4.3300e-04 - mean_squared_error: 4.3300e-04 - val_loss: 3.7085e-04 - val_mean_squared_error: 3.7085e-04
Epoch 88/100
4212/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.3342e-04 - mean_squared_error: 4.3342e-04
Epoch 88: val_loss did not improve from 0.00037
4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 4.3342e-04 - mean_squared_error: 4.3342e-04 - val_loss: 3.6706e-04 - val_mean_squared_error: 3.6706e-04
Epoch 89/100
4208/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.3437e-04 - mean_squared_error: 4.3437e-04
Epoch 89: val_loss did not improve from 0.00037
4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 4.3436e-04 -

4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 4.2806e-04 - mean_squared_error: 4.2806e-04 - val_loss: 3.6583e-04 - val_mean_squared_error: 3.6583e-04
Epoch 93/100
4219/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.3100e-04 - mean_squared_error: 4.3100e-04
Epoch 93: val_loss did not improve from 0.00037
4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 4.3100e-04 - mean_squared_error: 4.3100e-04 - val_loss: 3.6856e-04 - val_mean_squared_error: 3.6856e-04
Epoch 94/100
4206/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.2955e-04 - mean_squared_error: 4.2955e-04
Epoch 94: val_loss improved from 0.00037 to 0.00037, saving model to SRCNN.h5


4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 4.2956e-04 - mean_squared_error: 4.2956e-04 - val_loss: 3.6564e-04 - val_mean_squared_error: 3.6564e-04
Epoch 95/100
4216/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.3006e-04 - mean_squared_error: 4.3006e-04
Epoch 95: val_loss improved from 0.00037 to 0.00036, saving model to SRCNN.h5


4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 4.3006e-04 - mean_squared_error: 4.3006e-04 - val_loss: 3.6473e-04 - val_mean_squared_error: 3.6473e-04
Epoch 96/100
4218/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.3033e-04 - mean_squared_error: 4.3033e-04
Epoch 96: val_loss did not improve from 0.00036
4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 4.3033e-04 - mean_squared_error: 4.3033e-04 - val_loss: 3.6744e-04 - val_mean_squared_error: 3.6744e-04
Epoch 97/100
4207/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.2981e-04 - mean_squared_error: 4.2981e-04
Epoch 97: val_loss did not improve from 0.00036
4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 4.2981e-04 - mean_squared_error: 4.2981e-04 - val_loss: 3.6788e-04 - val_mean_squared_error: 3.6788e-04
Epoch 98/100
4212/4220 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.3076e-04 - mean_squared_error: 4.3076e-04
Epoch 98: val_loss did not improve from 0.00036
4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - loss: 4.3076e-04 -

4220/4220 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 4.2922e-04 - mean_squared_error: 4.2922e-04 - val_loss: 3.6312e-04 - val_mean_squared_error: 3.6312e-04
Eğitim tamamlandı!


In [ ]:
predict_image(
        model_path="/content/SRCNN.h5",
        image_path="/content/drive/MyDrive/2209/high_reso/test/val/M0501_img000010.jpg",
        output_folder="/content/drive/MyDrive/2209/high_reso/results"
    )

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model yüklendi: /content/SRCNN.h5
Bicubic upscaled görüntü kaydedildi: /content/drive/MyDrive/2209/high_reso/results/M0501_img000010_bicubic.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
SRCNN sonucu kaydedildi: /content/drive/MyDrive/2209/high_reso/results/M0501_img000010_srcnn.png
Bicubic PSNR: 33.28 dB
SRCNN PSNR: 33.84 dB
PSNR İyileştirmesi: 0.56 dB
